# 운동작업량 기반 피로도 예측 (2022 전남대학교 AI 활용 해커톤 최우수상)

당시 Google Colab에서 작성한 원본 코드에 **설명 주석만** 덧붙였습니다. 코드 동작은 원본과 같습니다.
자세한 배경과 한계는 README를 참고하세요.

## 1. 데이터 준비

선수 18164번의 기록 중, 피로도에 영향을 줄 수 있는 **기분 · 스트레스 수치가 비슷한 날 8일치**만 골랐습니다.
그래야 운동작업량과 피로도 사이의 관계만 볼 수 있다고 판단했습니다.

In [ ]:
import numpy as np  # 숫자 배열(벡터) 계산용 라이브러리

# X: 입력값 = 하루 운동작업량 (운동강도 × 운동시간), 8일치
X = np.array([792,240,272,637,80,86,518,356,])
X  # Colab에서는 셀 마지막 변수가 자동으로 출력됨

In [ ]:
print(X)  # 입력 데이터 확인

In [ ]:
# Y: 정답값 = 같은 날의 피로도 (선수가 기록한 주관적 점수)
# X의 각 값과 같은 순서로 짝을 이룸 (예: 작업량 792 → 피로도 5)
Y = np.array([5,7,7,4,5,6,6,6,])

In [ ]:
print(Y)  # 정답 데이터 확인

## 2. 모델 만들기

입력 1개(운동작업량) → 은닉층 노드 5개 → 출력 1개(피로도)인 작은 신경망입니다.

> **2026 회고:** 두 층 모두 활성화 함수가 없어서(기본값 linear), 층을 쌓아도 결국 `피로도 = a × 작업량 + b` 형태의 **선형 회귀와 같은 모델**입니다. 비선형 관계를 배우려면 `activation='relu'`가 필요합니다.

In [ ]:
from keras.models import Sequential  # 층을 순서대로 쌓는 모델
from keras.layers import Dense         # 모든 입력과 연결된 기본 층(완전연결층)

In [ ]:
model = Sequential()

# 은닉층: 입력값 1개(운동작업량)를 받아 노드 5개로 계산
# activation을 지정하지 않아 선형(linear) 계산만 함
model.add(Dense(units=5, input_dim=1))

# 출력층: 노드 1개 = 예측 피로도
model.add(Dense(units=1))

In [ ]:
model.summary()  # 층 구조와 학습할 가중치 개수 확인 (총 16개: 1×5+5, 5×1+1)

## 3. 학습

- **손실 함수 MSE:** 예측 피로도와 실제 피로도 차이의 제곱 평균. 작을수록 잘 맞춘 것
- **옵티마이저 Adam:** 손실이 줄어드는 방향으로 가중치를 조금씩 고치는 방법
- **epochs=1000:** 8개 데이터를 1,000번 반복해서 학습

최종 학습 오차(loss)는 약 **0.7241**이었습니다.

In [ ]:
model.compile(loss='mse', optimizer='adam')  # 손실 함수와 학습 방법 지정
model.fit(x=X, y=Y, epochs=1000)             # 8개 데이터로 1,000번 반복 학습

## 4. 예측

In [ ]:
# 학습에 쓴 8일치 작업량을 다시 넣어, 모델이 예측한 피로도를 확인
# (실제 Y와 비교하면 얼마나 맞췄는지 볼 수 있음)
Yhat_train = model.predict(X)

In [ ]:
print(Yhat_train)

In [ ]:
# 구글 드라이브 연결 (당시 파일 저장용으로 연결, 예측 자체에는 사용하지 않음)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 새로운 운동작업량 810을 넣었을 때의 예상 피로도
# → 감독이 "내일 이만큼 훈련하면 피로도가 얼마나 될까?"를 미리 볼 수 있음
Yhat_test1 = model.predict([810])

In [ ]:
print(Yhat_test1)

## 5. 그래프 (미완성)

> 당시 결과 그래프를 그리려다 멈춘 부분입니다. 원본 기록으로 남겨 두었으며, **그대로 실행하면 오류가 납니다.**
> 발표에 쓴 그래프는 README의 이미지를 참고하세요.

In [ ]:
import matplotlib.pyplot as plt  # 그래프 그리기 라이브러리
plt.figure(figsize=(8,6))        # 그래프 전체 크기 지정

# [오류] subplot의 세 번째 값은 칸 번호(정수)여야 하는데 배열 Y를 넣음
plt.subplot(2,2,Y) # (1) 첫 번째 공간에 axes 생

# [오류] 리스트 안에 배열 X가 섞여 있어 선 그래프로 그릴 수 없음
plt.plot([1,X,3],'red')
plt.subplot(2,2,4) # (2) 두 번째 공간에 axes 생성

### 고친다면 (참고용)

```python
plt.scatter(X, Y, label='실제 피로도')
xs = np.linspace(0, 900, 100)
plt.plot(xs, model.predict(xs), 'red', label='모델 예측')
plt.xlabel('운동작업량'); plt.ylabel('피로도'); plt.legend(); plt.show()
```